## Decision Localization

![Image](https://hackmd.io/_uploads/SkjywwKoWl.png)

*Image generated using ChatGPT*

## Introduction

The model can recognize objects in images, but it is not clear what exactly it focuses on when making decisions. Sometimes it detects the correct object, and sometimes it relies on random features such as background or colors. Therefore, we want to check which parts of the image have the greatest influence on its prediction.

## Task

You are given a trained binary classifier. Your task is to generate **saliency maps (heatmaps)** indicating the regions of the image that had the greatest influence on its decision.

## Data

The dataset is a subset of **COCO2017**, containing images where the class recognized by the model as positive is always present.

You will receive only the **validation set** – 1200 images.

All images have a size of **224 × 224 pixels**.

You are also given a **ResNet-18 model**, trained for a **binary classification task**. The model determines whether a specific object is present in the image.

The same model will be used during testing. The test set also consists of 1200 images.

## Evaluation Metric

Your solution will be evaluated based on the quality of the generated saliency maps.

For each heatmap, the **Intersection over Union (IoU)** metric is computed between:

* the region indicated by the heatmap (a soft region),
* the true object location defined by the COCO mask.

The IoU is defined as:

$$
IoU = \frac{|H \cap G|}{|H \cup G|}
$$

where:

* $H$ – the important region indicated by the model,
* $G$ – the ground truth object region from annotations.

The final score is calculated as:

$$
results_evaluation =
\begin{cases}
0 & \text{if IoU} \le 0.25 \\
100 \cdot \dfrac{\text{IoU} - 0.191}{0.245 - 0.191} & \text{if } 0.191 \leq \text{IoU}_{mean} \le 0.245 \\
100 & \text{otherwise}
\end{cases}
$$

## Constraints

* The solution will run **without internet access**
* A **GPU** is available
* Total evaluation time must not exceed **3 minutes**
* Model weights must not be modified
* Allowed libraries: `torch`, `torchvision`, `numpy`, `cv2`

Your solution will be tested on the competition platform without internet access and in a GPU-enabled environment.

## Submission Files

This notebook completed with your solution.


## Evaluation

Remember that during evaluation the flag `FINAL_EVALUATION_MODE` will be set to `True`.

For this task, you can earn between **0 and 100 points**. The number of points you receive will be calculated on a (hidden) test set on the competition platform, based on the formula described above, and then rounded to an integer.

If your solution does not meet the specified criteria or fails to run correctly, you will receive **0 points** for the task.


## Starter Code


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

import os
import random
import numpy as np
import cv2
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

FINAL_EVALUATION_MODE = False  # During evaluation, this flag will be set to True.


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

def download_data():
    """Downloads the dataset from Google Drive and saves it in the 'data' folder."""
    import shutil
    import gdown

    # Create or reset the 'data' folder
    if not os.path.exists('data'):
        os.makedirs('data')
    else:
        shutil.rmtree('data')
        os.makedirs('data')

    # Download the file from Google Drive and save it in the 'data' folder
    url_model = "https://drive.google.com/file/d/1b6WhjG_GDihBNKLIXtFcaclJKkTWbZZk/view?usp=drive_link"
    gdown.download(url_model, f'data/model.pth', quiet=True)

    url_data = "https://drive.google.com/file/d/1hVn9m0KRuE-0x7zpPYcwkxDFMNoaM-U9/view?usp=drive_link"
    gdown.download(url_data, f'data/val_data.npz', quiet=True)


if not FINAL_EVALUATION_MODE:
    download_data()

## Model Definition


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

# Model architecture definition.
class ResNet18Classifier(nn.Module):
    def __init__(self, pretrained=False, num_classes=1):
        super().__init__()

        if pretrained:
            weights = models.ResNet18_Weights.DEFAULT
        else:
            weights = None

        resnet = models.resnet18(weights=weights)

        self.encoder = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.encoder(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.head(x)
        return x

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

# Model initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_A = ResNet18Classifier(num_classes=1).to(device)

# Loading pretrained weights
model_A.load_state_dict(torch.load("data/model.pth", map_location=device))
model_A.eval()

## Data Loading


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

class CocoSegmentationDataset(Dataset):
    def __init__(self, npz_path, mode="val"):
        """
        npz_path: Path to .npz file (e.g. 'data/val_data.npz')
        mode: internal key inside npz
        """

        if not os.path.exists(npz_path):
            raise FileNotFoundError(f"Nie znaleziono pliku: {npz_path}")

        # Load data from NPZ
        data = np.load(npz_path, allow_pickle=True)
        
        # Key mapping
        key = f"{mode}_ds"
        if key not in data:
            raise KeyError(f"Plik {npz_path} nie zawiera klucza {key}")
            
        self.samples = data[key].tolist()
        
        # Define ImageNet transforms
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], 
            std=[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # Convert image (uint8 [H,W,C] -> float32 [C,H,W])
        # 'image' is your JPG from the original dataset
        img_np = sample["image"]
        img_tensor = torch.from_numpy(img_np).to(torch.float32) / 255.0
        img_tensor = img_tensor.permute(2, 0, 1) # Convert to [C, H, W]
        img_tensor = self.normalize(img_tensor)
        
        # Convert mask (uint8 [H,W] -> float32 [1,H,W])
        # 'mask' is your PNG mask
        mask_np = sample["mask"]
        mask_tensor = torch.tensor(mask_np / 255.0, dtype=torch.float32)
        
        # Add channel dimension if mask is 2D
        if mask_tensor.ndim == 2:
            mask_tensor = mask_tensor.unsqueeze(0)
            
        return img_tensor, mask_tensor

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

# Initialization
val_dataset = CocoSegmentationDataset("data/val_data.npz")
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=True)

## Your Solution


In [ ]:
# Your solution - make changes only here.
# Do not change function arguments or the number of returned values.
# The function has been filled with a sample solution, which is far from optimal -
# in particular, it does not use the models at all.
# It is intended as a starting point for building your solution.


def your_solution(img, model):
    _, H, W = img.shape

    # Randomly generated mask.
    mask = torch.randint(0, 2, (H, W), dtype=torch.float32)

    return mask

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

# Display a sample image from the dataset.
if not FINAL_EVALUATION_MODE:
    import matplotlib.pyplot as plt

    def plot_example(dataset, model, your_solution):
        img, mask = dataset[0]
        pred = your_solution(img, model)
        if torch.is_tensor(pred):
            pred = pred.detach().cpu().numpy()
        mask_to_show = mask.squeeze().cpu().numpy()

        # Permute channels    
        img_display = img.permute(1, 2, 0).numpy()
        # Undo normalization for the image
        img_display = img_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        # Clip values to [0, 1] for proper visualization
        img_display = np.clip(img_display, 0, 1)

        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        plt.title("Oryginał")
        plt.imshow(img_display)
        
        plt.subplot(1, 3, 2)
        plt.title("Maska (GT)")
        plt.imshow(mask_to_show, cmap='gray')
        
        plt.subplot(1, 3, 3)
        plt.title("Predykcja")
        plt.imshow(pred, cmap='jet', alpha=0.5)
        plt.imshow(img_display, alpha=0.5)
        plt.show()

    # Run visualization function showing a sample image from the dataset along with GT mask and prediction
    plot_example(val_dataset, model_A, your_solution)

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

# Computes IoU metric between two masks.

def compute_iou(pred, target):
    # Ensure tensors are binary (0 or 1)
    pred = (pred > 0.5).float()
    target = (target > 0.5).float()
    
    intersection = (pred * target).sum()
    union = (pred + target).clamp(0, 1).sum()
    return (intersection + 1e-8) / (union + 1e-8)

## Evaluation

Running the cell below will allow you to check how many points your solution would score on the training data.

Before submission, make sure that the entire notebook runs from start to finish **without errors** when the flag `FINAL_EVALUATION_MODE = True`, and **without any user interaction** after selecting the **"Run All"** option.


In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################]

# Computing participant score (explicit, uses val_loader)
def compute_score(loader, model):
    model.eval()
    ious = []
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for img, mask in loader:
            img, mask = img.to(device), mask.to(device)
            
            # Pass a single image (C, H, W) to participant solution
            # Assume your_solution returns mask (H, W) or (1, H, W)
            pred = your_solution(img[0], model)
            mask = mask.to(pred.device)
            
            # Match shapes for IoU
            ious.append(compute_iou(pred.squeeze(), mask.squeeze()))
    
    mIoU = sum(ious) / len(ious)
    
    # Score scaling: 0.191 -> 0 pts, 0.245 -> 100 pts
    raw_score = 100 * (mIoU - 0.191) / (0.245 - 0.191)
    score = int(torch.clamp(raw_score, min=0, max=100).round().item())
    
    return mIoU, score

In [ ]:
######################### DO NOT MODIFY THIS CELL ##########################

# Evaluates your solution on validation set by computing mean IoU.
# A similar score should be obtained on the hidden test set.

if not FINAL_EVALUATION_MODE:
    mIoU, score = compute_score(val_loader, model_A)
    print(f"mIoU: {mIoU:.4f} | Estymowana liczba punktów: {score}/100")